# DistilBERT extension: data and tokenization

Prepare the existing GoEmotions sentiment split for DistilBERT. This uses raw `text`, the same 680/120 train/validation split as the neural experiments, and leaves the held-out test split untouched until evaluation. Run from the repository root in Colab or another environment with the project files available.

In [ ]:
%pip install -q "transformers>=4.45,<6.0" "torch>=2.4,<3.0" pandas scikit-learn

from pathlib import Path
import math
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT_DIR = next((path for path in candidates if (path / 'src' / 'preprocessing.py').exists()), None)
if ROOT_DIR is None:
    raise FileNotFoundError('Start from the repository root or place the repository in the current directory.')
sys.path.insert(0, str(ROOT_DIR))

from src.preprocessing import SEED, load_splits, train_val_split
from src.models import build_transformer
from transformers import AutoTokenizer
import numpy as np

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH_CAP = 64
SEED

## Load the shared split

`load_splits` defaults to cleaned text, so pass `text_col='text'` to give DistilBERT the original comments. Only the training portion determines the validation split and token-length limit.

In [ ]:
X_train_all, y_train_all, X_test, y_test = load_splits(text_col='text')
X_train, X_val, y_train, y_val = train_val_split(X_train_all, y_train_all, seed=SEED)
print(f'train={len(X_train)}, validation={len(X_val)}, test={len(X_test)}')
print(f'train labels={np.bincount(y_train)}, validation labels={np.bincount(y_val)}')
assert (len(X_train), len(X_val), len(X_test)) == (680, 120, 200)
assert set(np.unique(y_train_all)) == {0, 1}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_lengths = [len(tokenizer.encode(text, add_special_tokens=True)) for text in X_train]
p99_length = int(math.ceil(np.quantile(train_lengths, 0.99)))
MAX_LENGTH = MAX_LENGTH_CAP if p99_length <= MAX_LENGTH_CAP else p99_length
train_truncation_rate = float(np.mean(np.asarray(train_lengths) > MAX_LENGTH))
print(f'train subword lengths: p99={p99_length}, max={max(train_lengths)}')
print(f'max_length={MAX_LENGTH}; training truncation rate={train_truncation_rate:.1%}')
if p99_length > MAX_LENGTH_CAP:
    print('The training p99 exceeds 64 tokens, so max_length was increased to cover the training p99.')

## Tokenize without fitting on test data

The pretrained tokenizer is fixed, so no vocabulary is learned from this dataset. The maximum sequence length is selected from training comments only. Test comments are tokenized with that fixed setting for later evaluation; they are not used to choose it.

In [ ]:
def tokenize_texts(texts):
    return tokenizer(
        list(texts),
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
        return_attention_mask=True,
    )

tokenized_train = tokenize_texts(X_train)
tokenized_val = tokenize_texts(X_val)
tokenized_test = tokenize_texts(X_test)
for name, batch in [('train', tokenized_train), ('validation', tokenized_val), ('test', tokenized_test)]:
    ids = np.asarray(batch['input_ids'])
    masks = np.asarray(batch['attention_mask'])
    assert ids.shape == masks.shape == (len(batch['input_ids']), MAX_LENGTH)
    print(f'{name}: input_ids={ids.shape}, attention_mask={masks.shape}')

model = build_transformer(MODEL_NAME)
print(f'{model.config._name_or_path}: {model.config.num_labels} labels, {model.num_parameters():,} parameters')

## Prepared inputs

`tokenized_train`, `tokenized_val`, and `tokenized_test` contain `input_ids` and `attention_mask`; labels are in the matching `y_train`, `y_val`, and `y_test` arrays. J2 will add fine-tuning and metrics. Do not select a checkpoint or hyperparameter from test results.